In [ ]:
import torch
import time

# 配置参数
M, N, K = 3000, 2048, 5000  # 矩阵尺寸 (M, K) @ (K, N)
NUM_RUNS = 100               # 运行次数

def benchmark_matrix_mul(device='cpu', num_runs=1000):
    try:
        # 初始化矩阵（统一在CPU创建）
        A_cpu = torch.randn(M, K)
        B_cpu = torch.randn(K, N)
        
        # 数据转移计时
        transfer_start = time.time()
        A = A_cpu.to(device)
        B = B_cpu.to(device)
        if device == 'cuda':
            torch.cuda.synchronize()
        transfer_time = time.time() - transfer_start
        
        # 预热（不纳入计时）
        _ = torch.mm(A[:10, :10], B[:10, :10])
        if device == 'cuda':
            torch.cuda.synchronize()
        
        # 正式计时
        compute_start = time.time()
        for _ in range(num_runs):
            C = torch.mm(A, B)  # 实际计算
            if device == 'cuda':
                torch.cuda.synchronize()  # 确保每次计算完成
        total_compute_time = time.time() - compute_start
        
        # 验证最后一次结果
        checksum = C.norm().item()
        return transfer_time, total_compute_time, checksum
    
    except RuntimeError as e:
        if 'CUDA out of memory' in str(e):
            print(f"内存不足！请尝试减小矩阵尺寸（当前：{M}x{K} @ {K}x{N}")
            print("建议调整 M, N, K = 10000, 200, 10000")
        else:
            print(f"运行时错误：{str(e)}")
        return None

# 执行测试
print(f"正在测试 {NUM_RUNS} 次矩阵乘法 ({M}x{K} @ {K}x{N})...")

# CPU测试
cpu_transfer, cpu_time, cpu_check = benchmark_matrix_mul('cpu', NUM_RUNS)
if cpu_time:
    print(f"\nCPU结果:")
    print(f"  - 数据传输: {cpu_transfer:.4f}s")
    print(f"  - 总计算时间: {cpu_time:.2f}s")
    print(f"  - 单次平均时间: {cpu_time/NUM_RUNS:.4f}s")
    print(f"  - 结果校验值: {cpu_check:.2f}")

# CUDA测试
if torch.cuda.is_available():
    cuda_result = benchmark_matrix_mul('cuda', NUM_RUNS)
    if cuda_result:
        cuda_transfer, cuda_time, cuda_check = cuda_result
        print(f"\nCUDA结果:")
        print(f"  - 数据传输: {cuda_transfer:.4f}s")
        print(f"  - 总计算时间: {cuda_time:.2f}s")
        print(f"  - 单次平均时间: {cuda_time/NUM_RUNS:.6f}s")
        print(f"  - 结果校验值: {cuda_check:.2f}")
        print(f"\n性能对比:")
        print(f"  总加速比: {cpu_time / cuda_time:.1f}x")
        print(f"  单次加速比: {(cpu_time/NUM_RUNS) / (cuda_time/NUM_RUNS):.1f}x")
else:
    print("\nCUDA不可用")

# 内存清理
if torch.cuda.is_available():
    torch.cuda.empty_cache()

正在测试 100 次矩阵乘法 (3000x5000 @ 5000x2048)...

CPU结果:
  - 数据传输: 0.0000s
  - 总计算时间: 6.95s
  - 单次平均时间: 0.0695s
  - 结果校验值: 175234.62

CUDA结果:
  - 数据传输: 0.1331s
  - 总计算时间: 0.69s
  - 单次平均时间: 0.006924s
  - 结果校验值: 175262.58

性能对比:
  总加速比: 10.0x
  单次加速比: 10.0x
